## Monte-Carlo-Simulation zur finanzmathematischen Modellierung der Call-Option mithilfe des Cox-Ross-Rubinstein-Modells

#### Das Cox-Ross-Rubinstein-Modell (CRR-Modell) ist ein sehr einfaches Modell zur Bewertung von Optionen.
#### Es ist im Grunde ein diskretes Binomialmodell für Aktienkurse.

#### Hintergrund:

##### In jedem kleinen Zeitschritt kann der Aktienkurs nur:

##### steigen um Faktor u
##### oder fallen um Faktor d.

##### Ist X_t der Aktienpreis zum Zeitpunkt t, so beträgt im steigenden Kurs der neue Aktienpreis X_(t+1) zum Zeitpunkt (t+1): X_t * u.
##### Umgekehrt beträgt der neue Aktienpreis X_(t+1) dann: X_t * d.

##### Hierbei ist 0 < d < u.

##### Das passiert über viele mehrere Schritte im Binomialverteilungsmodell.
##### Dabei zählen wir die Anzahl der gestiegenen Preise auf.

##### Im CRR-Modell benutzen wir die risikoneutrale Wahrscheinlichkeit

##### q = (1+r-d)/(u-d) aus ]0,1[, mit Zinssatz r für die Modellierung des Preises B_t = (1+r)^t eines festverzinslichen Wertpapiers.

##### Das ist die risikoneutrale Wahrscheinlichkeit, sodass Preise korrekt gewählt sind.

### Wir simulieren den Optionspreis der Call-Option von einer Aktie (etwa Microsoft)

#### Beachte hierzu: T = 100 Zeitpunkte, r = 1 % der risikolose Zinssatz des festverzinslichen Wertpapiers, X_0 = 10 der Anfangspreis der Aktie Microsoft.

#### Die Call-Option wird hier dargestellt als max(0,X_T - 10), wobei 10 für den Ausübungspreis gilt.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

### Wir veranschaulichen uns zunächst die zufällig generierten Preismultiplikatoren.

In [2]:
n = 1000 # Anzahl Wiederholungen
T = 100 # Anzahl Preismultiplikatoren
u = 1.5
d = 0.8
sim_preisverlauf = np.random.choice([u, d], size=(n,T))
sim_preisverlauf

array([[1.5, 1.5, 1.5, ..., 0.8, 1.5, 1.5],
       [1.5, 0.8, 0.8, ..., 0.8, 1.5, 0.8],
       [0.8, 0.8, 1.5, ..., 0.8, 0.8, 1.5],
       ...,
       [1.5, 1.5, 0.8, ..., 1.5, 1.5, 0.8],
       [1.5, 1.5, 0.8, ..., 1.5, 0.8, 0.8],
       [0.8, 1.5, 0.8, ..., 0.8, 0.8, 1.5]])

### Anschließend modellieren wir den Preis der Call-Option, um den heutigen Wert des Call-Optionspreises in 100 zukünftigen Zeitpunkten zu bestimmen.
### Wir wenden hierauf die Monte-Carlo-Simulation an, um den Optionspreis zu simulieren. Dies erfolgt durch die Approximation des Erwartungswertes durch das arithmetische Mittel der Auszahlungen für die erzeugten Preisverläufe (auch genannt als direkte Simulation).

In [3]:
K = 10
X_0 = 10
r = 0.01
q = (1+r-d)/(u-d)
# L = Anzahl Aufwärtsbewegungen; wir erzeugen hier jedoch 1000 zufällige solcher binomialverteilten Zufallsvariablen
L = np.random.binomial(T, q, size = n)

# Endkurs
X_T = X_0 * u**L * d**(T-L)
#print('1000 generierte Endkurse von X_T =\n', X_T)
# Call-Auszahlungen
Call_Opt = np.maximum(X_T - K, 0)
print('1000 generierte Call_Optionen über X_T =\n', Call_Opt)
print('')
# Diskontierungsfaktor
disc = 1 / (1+r)**T

# Approximation des Erwartungswertes (Optionspreis) disc * Erwart.[Call_Opt] mithilfe der direkten Simulation
E_Call_Opt = disc * np.mean(Call_Opt)

print('Der ungefähre Optionspreis zum Auszahlungszeitpunkt T = 100 beträgt=\n', E_Call_Opt)

1000 generierte Call_Optionen über X_T =
 [0.00000000e+00 0.00000000e+00 3.81999999e+01 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 3.71022220e+00 1.57066666e+01 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 3.07724609e+02 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 1.57066666e+01
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 3.81999999e+01 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 3.81999999e+01 1.59453125e+02 0.00000000e+00 3.71022220e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 5.85733642e+

### Wir evaluieren die Methode, indem wir den Prozess 100 mal wiederholen und hiervon den Durchschnittswert berechnen.

In [4]:
m = 100
Opt_preise = np.zeros(m)

for i in range(m):
    L = np.random.binomial(T, q, size=m)
    X_T = X_0 * u**L * d**(T - L)
    Call_Opt = np.maximum(X_T - K, 0)
    Opt_preise[i] = disc * np.mean(Call_Opt)

print('Preise der Call-Option =\n', Opt_preise)
print('')
print('Durchschnittswert der Preise der Call-Option=\n', np.mean(Opt_preise))

Preise der Call-Option =
 [ 5.14970891 12.71767664  1.85030936  3.29880365  5.34968276  0.83378217
 16.79683426 13.17635077  7.51286256  0.79263085  6.96376503  3.30877557
  1.13168689  1.12712178  0.8621953   2.18067487  1.3548339  15.48325357
  3.13456764  0.9508997   2.31150951 10.97508998  2.78870275 14.63871234
  5.96983393  3.2789982   9.25040974  3.83984212  0.97140388  0.46952667
 52.76727967  7.72587866 11.40239309 20.77486702  5.39305605 16.2437767
 17.08041785  7.10191171  7.4096377   1.97124267  1.69770572  0.62310922
  3.43621756  6.44726065  1.42427717  0.84847819 26.29950093 13.48721768
 25.63975112  1.95074079 11.43622297 11.52352533  1.7470845   1.68822473
  1.84415044  2.18769153  6.03527297  1.71565895  3.44358728  4.72953159
 10.88872924  4.37940564  0.48187954  5.23490838  0.28480251  0.81920733
  7.44405075 17.18455105  4.55921392  3.27880975  1.34493257  1.65802703
  2.24353887  0.69255249  1.83024487  7.57339062  2.07229994  1.0217072
  4.84128796  5.90118844  1

### Fazit:
#### Geht man nur von diesen 100-Wiederholungen aus, so ist heute im Schnitt der zukünftige Optionspreis nicht wertvoller als der jetzige Ausübungspreis von K = 10. Dies stellt jedoch keine optimale Investitionsentscheidung dar, sodass weitere geeignetere Evaluierungsmethoden betrachtet werden müssen. 